# Oversold + High Volume Bounce on SPY
## Strategy Brief
This strategy aims to capitalize on mean reversion by identifying oversold conditions in the SPY ETF, combined with a spike in trading volume, which often precedes a bounce back in price. The signal is generated when the Relative Strength Index (RSI) indicates oversold conditions (below 30) and volume is significantly higher than its average. The trade logic involves entering a long position when these conditions are met and exiting when the RSI returns to neutral levels. Historical backtesting suggests this approach can outperform a simple buy-and-hold strategy by capturing short-term reversals.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for our strategy, such as the lookback periods for RSI and volume, and the thresholds for entering and exiting trades.

In [ ]:
RSI_PERIOD = 14
VOLUME_LOOKBACK = 20
RSI_OVERSOLD = 30
RSI_EXIT = 50
VOLUME_THRESHOLD = 1.5

### PHASE 2 - Data Exploration
We will download historical SPY data from Yahoo Finance, compute the RSI and volume indicators, and visualize them alongside the price data to understand their behavior.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema

# Download SPY data
data = yf.download('SPY', start='2010-01-01')

# Calculate RSI
delta = data['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))
data['RSI'] = rsi

# Calculate Volume SMA
volume_sma = data['Volume'].rolling(window=VOLUME_LOOKBACK).mean()
data['Volume_SMA'] = volume_sma

# Plotting
plt.figure(figsize=(14, 7))
plt.subplot(3, 1, 1)
plt.plot(data['Close'], label='Close Price')
plt.title('SPY Price with RSI and Volume')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(data['RSI'], label='RSI', color='orange')
plt.axhline(RSI_OVERSOLD, color='red', linestyle='--')
plt.axhline(RSI_EXIT, color='green', linestyle='--')
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(data['Volume'], label='Volume', color='gray')
plt.plot(data['Volume_SMA'], label='Volume SMA', color='blue')
plt.legend()

plt.tight_layout()
plt.show()

### PHASE 3 - Strategy Engineering
We will create a signal based on the RSI and volume conditions. The strategy enters a long position when RSI is below the oversold threshold and volume is above the threshold.

In [ ]:
data['Signal'] = 0

# Generate signal
oversold_condition = (data['RSI'] < RSI_OVERSOLD)
high_volume_condition = (data['Volume'] > VOLUME_THRESHOLD * data['Volume_SMA'])

# Enter signal
data.loc[oversold_condition & high_volume_condition, 'Signal'] = 1

# Exit signal
exit_condition = (data['RSI'] > RSI_EXIT)
data.loc[exit_condition, 'Signal'] = 0

# Positions
positions = data['Signal'].shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the generated signals and plot the resulting equity curve.

In [ ]:
data['Daily_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = positions * data['Daily_Return']
data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(12, 6))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.plot((1 + data['Daily_Return']).cumprod(), label='Buy and Hold Equity Curve', linestyle='--')
plt.legend()
plt.title('Equity Curve')
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance by calculating key metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and maximum drawdown, and compare these against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve, daily_returns):
    total_return = equity_curve.iloc[-1] - 1
    n_years = len(equity_curve) / 252
    cagr = (equity_curve.iloc[-1]) ** (1 / n_years) - 1
    sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
    downside_returns = daily_returns[daily_returns < 0]
    sortino_ratio = np.mean(daily_returns) / np.std(downside_returns) * np.sqrt(252)
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(data['Equity_Curve'], data['Strategy_Return'])
buy_and_hold_metrics = calculate_performance_metrics((1 + data['Daily_Return']).cumprod(), data['Daily_Return'])

comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})

print(comparison_table)

### PHASE 6 - Deploy & Monitor
We will create a function that fetches the last 60 days of SPY data, computes today's signal, and prints the recommended position.

In [ ]:
def get_today_signal():
    recent_data = yf.download('SPY', period='60d')
    
    # Calculate RSI
    delta = recent_data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    recent_data['RSI'] = rsi
    
    # Calculate Volume SMA
    volume_sma = recent_data['Volume'].rolling(window=VOLUME_LOOKBACK).mean()
    recent_data['Volume_SMA'] = volume_sma
    
    # Determine today's signal
    oversold_condition = (recent_data['RSI'].iloc[-1] < RSI_OVERSOLD)
    high_volume_condition = (recent_data['Volume'].iloc[-1] > VOLUME_THRESHOLD * recent_data['Volume_SMA'].iloc[-1])
    
    if oversold_condition and high_volume_condition:
        print('Signal: Enter Long Position')
    else:
        print('Signal: No Position')

get_today_signal()